# Multilingual Topic Modeling for Philippine Student Feedback

Notebook equivalent of the `../topic-modeling.faculytics/` CLI experiment.

**Pipeline:** `load → preprocess → LaBSE embed → BERTopic (UMAP+HDBSCAN+KeyBERTInspired) → evaluate → save`

**Datasets:** drop one of `feedback_augmented_v1.json`, `uc_dataset_20krows1.csv`, or `uc_dataset_filtered.json` into `./data/`. See README for details.

**Runtime:** ~5 min on GPU for the augmented dataset (first run downloads LaBSE ~2 GB and computes embeddings; cached after). ~15–20 min on the real dataset.

**Reproducibility:** seeded with `random_state=42` throughout. Heavy logic lives in `lib.py` so this notebook reads as narrative.

In [ ]:
# ── Colab bootstrap (no-op outside Colab) ───────────────────────────
# Detects Google Colab, installs the deps that aren't pre-installed there,
# and exposes `IN_COLAB` so later cells pick the right Plotly renderer.
# Safe to run anywhere; the install step is skipped off-Colab.
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Detected Google Colab — installing dependencies (~1–2 min on first run)...")
    !pip install -q bertopic sentence-transformers umap-learn hdbscan gensim
    print("✓ dependencies installed — make sure lib.py and your dataset are uploaded too")
else:
    print("Not running in Colab — skipping pip install (use requirements.txt locally).")


## 1. Setup

In [ ]:
import gc
import json
import logging
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
from tqdm.auto import tqdm

import lib

pio.renderers.default = "colab" if IN_COLAB else "notebook"
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
sns.set_style("whitegrid")
pd.set_option("display.max_colwidth", 120)

In [ ]:
# ── Run configuration — edit these to control the run ────────────────
RUN_ID = "nb_015_real_filtered"
DATASET = "real_filtered"   # one of: "augmented", "real", "real_filtered"
SEED = 42

# Paths (resolved relative to lib.py — i.e. this notebook's folder)
DATA_DIR = lib.DATA_DIR
RUNS_DIR = lib.RUNS_DIR
FIGURES_DIR = lib.FIGURES_DIR
RUN_DIR = RUNS_DIR / f"run_{RUN_ID}"
FIG_DIR = FIGURES_DIR / f"run_{RUN_ID}"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Seed everything
random.seed(SEED)
np.random.seed(SEED)

print(f"RUN_ID  = {RUN_ID}")
print(f"DATASET = {DATASET}")
print(f"DATA_DIR = {DATA_DIR}")
print(f"RUN_DIR  = {RUN_DIR}")

In [ ]:
# ── Data presence check ──────────────────────────────────────────────
DATASET_FILES = {
    "augmented":     lib.AUGMENTED_DATASET_PATH,
    "real":          lib.UC_DATASET_PATH,
    "real_filtered": lib.UC_FILTERED_DATASET_PATH,
}
expected = DATASET_FILES[DATASET]

if expected.exists():
    print(f"✓ found {expected.name} ({expected.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"✗ missing: {expected}")
    print("  Drop the dataset file into ./data/, or uncomment the helper cell below to copy from a source path.")

# ── Optional helper: copy from a SOURCE_DIR ──────────────────────────
# Uncomment, set SOURCE_DIR, and run this cell once if your dataset files
# live elsewhere on disk (e.g. inside the original repo's data folder).
#
# import shutil
# SOURCE_DIR = Path("/path/to/your/dataset/folder")
# for fname in ["feedback_augmented_v1.json", "uc_dataset_20krows1.csv", "uc_dataset_filtered.json"]:
#     src = SOURCE_DIR / fname
#     dst = DATA_DIR / fname
#     if src.exists() and not dst.exists():
#         shutil.copy2(src, dst)
#         print(f"copied {fname}")
#     elif not src.exists():
#         print(f"not in source: {fname}")
#     else:
#         print(f"already present: {fname}")

### Sentiment-gated dataset (optional)

Mirrors the CLI's `prepare_dataset.py`. Builds `uc_dataset_filtered.json` from the raw UC CSV by:

- **Always keeping** negative + neutral feedback
- **Excluding** positive feedback shorter than `min_words` (default 10) — mostly generic praise ("good", "very good teacher") that forms a low-signal positive cluster

Auto-runs only when `DATASET == "real_filtered"` and the file is missing. Call `build_sentiment_gated(overwrite=True, min_words=...)` directly to regenerate or sweep thresholds.

In [ ]:
# ── Sentiment gating: build uc_dataset_filtered.json from the raw UC CSV ─
# Negative + neutral always included; positive only when >= min_words.
# Mirrors topic-modeling.faculytics/scripts/prepare_dataset.py.

LABEL_POSITIVE, LABEL_NEGATIVE, LABEL_NEUTRAL = 1.0, -1.0, 0.0


def build_sentiment_gated(
    source: Path = lib.UC_DATASET_PATH,
    output: Path = lib.UC_FILTERED_DATASET_PATH,
    min_words: int = 10,
    overwrite: bool = False,
) -> Path:
    if output.exists() and not overwrite:
        print(f"✓ {output.name} already exists ({output.stat().st_size / 1e6:.1f} MB) — pass overwrite=True to regenerate")
        return output
    if not source.exists():
        raise FileNotFoundError(f"raw UC dataset not found: {source}")

    df = pd.read_csv(source)
    df["label"] = pd.to_numeric(df["label"], errors="coerce")
    df = df.dropna(subset=["comment"])
    df["wc"] = df["comment"].astype(str).str.split().str.len()

    total = len(df)
    pos_short = df[(df["label"] == LABEL_POSITIVE) & (df["wc"] < min_words)]
    pos_long  = df[(df["label"] == LABEL_POSITIVE) & (df["wc"] >= min_words)]
    neg       = df[df["label"] == LABEL_NEGATIVE]
    neu       = df[df["label"] == LABEL_NEUTRAL]

    print(f"label breakdown (total {total:,}):")
    print(f"  positive short (<{min_words}w) — EXCLUDED: {len(pos_short):,} ({len(pos_short)/total:.1%})")
    print(f"  positive long  (>={min_words}w) — kept   : {len(pos_long):,} ({len(pos_long)/total:.1%})")
    print(f"  negative                       — kept   : {len(neg):,} ({len(neg)/total:.1%})")
    print(f"  neutral                        — kept   : {len(neu):,} ({len(neu)/total:.1%})")

    keep = pd.concat([pos_long, neg, neu])
    records = [
        {"text": row["comment"], "label": str(row["label"]), "lang_type": "mixed"}
        for _, row in keep.iterrows()
    ]
    output.write_text(json.dumps(records, ensure_ascii=False, indent=2))
    print(f"\n✅ saved {len(records):,} entries → {output}")
    return output


# Auto-build when the user picked real_filtered but the file is missing
if DATASET == "real_filtered" and not lib.UC_FILTERED_DATASET_PATH.exists():
    build_sentiment_gated()


In [ ]:
import torch

DEVICE = lib.DEVICE
print(f"device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Load Dataset

Three options:

- **`augmented`** — ~7k curated multilingual entries with `{text, label, lang_type}`. Best for baseline & tuning.
- **`real`** — ~34k raw UC student feedback (`comment`, `label` columns). Real-world data with noise.
- **`real_filtered`** — sentiment-gated real data (positive < 10 words excluded). Production baseline matching CLI Run 013.

`lang_type` is only available on the augmented dataset (visualization cells will fall back gracefully if absent).

In [ ]:
labels_raw: list | None = None
lang_type_raw: list | None = None

if DATASET == "augmented":
    with open(expected, encoding="utf-8") as f:
        data = json.load(f)
    texts_raw = [item["text"] for item in data]
    labels_raw = [item.get("label") for item in data]
    lang_type_raw = [item.get("lang_type") for item in data]
elif DATASET == "real":
    df = pd.read_csv(expected)
    text_col = "comment" if "comment" in df.columns else df.select_dtypes(include=["object"]).columns[0]
    df = df.dropna(subset=[text_col])
    texts_raw = df[text_col].astype(str).tolist()
    if "label" in df.columns:
        labels_raw = df["label"].tolist()
elif DATASET == "real_filtered":
    with open(expected, encoding="utf-8") as f:
        data = json.load(f)
    texts_raw = [item["text"] for item in data]
    labels_raw = [item.get("label") for item in data]
    lang_type_raw = [item.get("lang_type") for item in data]
else:
    raise ValueError(f"unknown DATASET={DATASET!r}")

print(f"loaded {len(texts_raw):,} raw texts")

In [ ]:
# Raw stats + samples
lengths = pd.Series([len(t.split()) for t in texts_raw], name="word_count")
print("word-count stats (raw):")
print(lengths.describe().round(1))

print("\n5 random samples:")
rng = np.random.default_rng(SEED)
for i in rng.choice(len(texts_raw), size=min(5, len(texts_raw)), replace=False):
    print(f"  [{i}] {texts_raw[i][:200]!r}")

if lang_type_raw is not None:
    print("\nlang_type distribution:")
    print(pd.Series(lang_type_raw).value_counts())

In [ ]:
# Pre-clean visualization
fig, axes = plt.subplots(1, 2 if lang_type_raw is not None else 1, figsize=(14, 4))
ax_hist = axes[0] if lang_type_raw is not None else axes

ax_hist.hist(lengths.clip(upper=100), bins=50, color="#4C72B0", edgecolor="white")
ax_hist.set_xlabel("word count (clipped at 100)")
ax_hist.set_ylabel("frequency")
ax_hist.set_title(f"raw word-count distribution — {DATASET} ({len(texts_raw):,} texts)")
ax_hist.axvline(3, color="crimson", linestyle="--", linewidth=1, label="<3 words → drop")
ax_hist.legend()

if lang_type_raw is not None:
    counts = pd.Series(lang_type_raw).value_counts()
    axes[1].barh(counts.index[::-1], counts.values[::-1], color="#55A868")
    axes[1].set_xlabel("count")
    axes[1].set_title("lang_type distribution")

plt.tight_layout()
plt.show()

## 3. Preprocess

Drops applied (in order; first match wins):

- `not_string` — input is not a `str` (e.g., NaN)
- `empty` — empty after `strip()`
- `excel_artifact` — `#NAME?`, `#VALUE!`, `#REF!`, `#DIV/0!`, `#NULL!`, `#NUM!`, `#N/A`
- `empty_after_clean` — became empty after URL / broken-emoji / laughter / repeated-char / punctuation-spam stripping
- `keyboard_mash` — only Latin-keyboard letters, no spaces, < 15% vowels
- `too_short` — < 3 words after cleaning

Non-destructive transforms applied to kept entries: URLs stripped, Unicode replacement chars (`\ufffd`) removed, laughter spam (`hahaha`/`lol`/etc.) stripped, repeated characters collapsed (`sooo` → `so`), punctuation spam reduced (`???` → `?`), whitespace normalized.

In [ ]:
texts, kept_indices = lib.clean_dataset_with_indices(texts_raw)

kept = len(texts)
dropped = len(texts_raw) - kept
drop_pct = dropped / len(texts_raw) * 100
print(f"kept    : {kept:,}")
print(f"dropped : {dropped:,} ({drop_pct:.1f}%)")

In [ ]:
# Drop-reason audit — surfaces *why* entries were dropped
reasons = [lib.clean_text_with_reason(t)[1] for t in texts_raw]
reason_counts = pd.Series(reasons).value_counts()
print(reason_counts)

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["#55A868" if r == "kept" else "#C44E52" for r in reason_counts.index]
ax.barh(reason_counts.index[::-1], reason_counts.values[::-1], color=colors[::-1])
ax.set_xlabel("count")
ax.set_title(f"drop-reason audit — {DATASET}")
for i, (label, val) in enumerate(zip(reason_counts.index[::-1], reason_counts.values[::-1])):
    pct = val / len(texts_raw) * 100
    ax.text(val, i, f"  {val:,} ({pct:.1f}%)", va="center", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Pre vs post word-count distribution
post_lengths = pd.Series([len(t.split()) for t in texts])

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
axes[0].hist(lengths.clip(upper=100), bins=50, color="#4C72B0", edgecolor="white")
axes[0].set_title(f"before cleaning ({len(texts_raw):,} texts)")
axes[0].set_xlabel("word count (clipped at 100)")
axes[0].set_ylabel("frequency")
axes[0].axvline(3, color="crimson", linestyle="--", linewidth=1)

axes[1].hist(post_lengths.clip(upper=100), bins=50, color="#55A868", edgecolor="white")
axes[1].set_title(f"after cleaning ({len(texts):,} texts)")
axes[1].set_xlabel("word count (clipped at 100)")
axes[1].axvline(3, color="crimson", linestyle="--", linewidth=1)

plt.tight_layout()
plt.show()

In [ ]:
# Sample inspection — 10 cleaned texts (with labels if available)
sample_idx = rng.choice(len(texts), size=min(10, len(texts)), replace=False)
rows = []
for i in sample_idx:
    orig_i = kept_indices[i]
    row = {"text": texts[i][:160]}
    if labels_raw is not None:
        row["label"] = labels_raw[orig_i]
    if lang_type_raw is not None:
        row["lang_type"] = lang_type_raw[orig_i]
    rows.append(row)
pd.DataFrame(rows)

## 4. Embed with LaBSE

[LaBSE](https://huggingface.co/sentence-transformers/LaBSE) is a multilingual sentence encoder trained on 109 languages. It outputs 768-dim L2-normalized embeddings well-suited for cross-lingual semantic similarity — critical for code-switched Cebuano/Tagalog/English feedback where word-level co-occurrence breaks down.

**First run:** ~2 GB model download + ~5 min encoding on a GPU (longer on CPU). **Subsequent runs:** load from `data/embeddings_cache_<DATASET>.npy` (instant).

Set `force=True` in the embed cell to regenerate (e.g., after cleaning logic changes).

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer(lib.LABSE_MODEL, device=DEVICE)
print(f"loaded {lib.LABSE_MODEL} on {DEVICE}")

In [ ]:
embeddings = lib.embed_texts(
    texts,
    embed_model=embed_model,
    batch_size=64,
    dataset_name=DATASET,
    force=False,
)
print(f"embeddings shape: {embeddings.shape}, dtype: {embeddings.dtype}")

### LaBSE embedding space (pre-clustering)

A 2D projection of the 768-dim LaBSE embeddings — what BERTopic "sees" before any clustering. Each dot is one document; colors mark `lang_type` when available.

**What to look for:**
- Documents with similar meaning should mix across languages — if the colored points are well-blended rather than forming language-only blobs, LaBSE is doing its multilingual job.
- Visible dense regions hint at natural topics; a smooth uniform cloud suggests clustering will be hard.

This UMAP is independent of the clustering UMAP (`n_components=2` here vs `n_components=10` in BERTopic).

In [ ]:
# 2D projection for visualization (separate from clustering UMAP — n_components=2)
from umap import UMAP

umap_2d = UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=SEED)
reduced_2d = umap_2d.fit_transform(embeddings)

df_2d = pd.DataFrame({
    "x": reduced_2d[:, 0],
    "y": reduced_2d[:, 1],
    "text": [t[:120] + ("..." if len(t) > 120 else "") for t in texts],
})
if lang_type_raw is not None:
    df_2d["lang_type"] = [lang_type_raw[kept_indices[i]] for i in range(len(texts))]
    color = "lang_type"
else:
    color = None

fig = px.scatter(
    df_2d, x="x", y="y", color=color, hover_data=["text"],
    title=f"LaBSE embeddings (UMAP 2D) — {DATASET} ({len(texts):,} docs)",
    opacity=0.6, height=500,
)
fig.update_traces(marker=dict(size=4))
fig.show()

## 5. Topic Modeling (BERTopic)

Pipeline (Grootendorst 2022):

1. **UMAP** — reduce 768-dim LaBSE embeddings to a low-dim space where density-based clustering is meaningful (curse-of-dimensionality mitigation).
2. **HDBSCAN** — density-based clustering. Documents that don't fit any cluster become topic `-1` (outliers).
3. **CountVectorizer** — bag-of-words for c-TF-IDF computation. Multilingual stop word list (~180 entries) suppresses role/function/filler words.
4. **KeyBERTInspired** — selects keywords by cosine similarity to cluster centroid in LaBSE space (replaces frequency-based c-TF-IDF). Language-agnostic; locked in at CLI Run 011 after the c-TF-IDF stop-word treadmill plateaued.

All parameters live in the cell below — edit and re-run from cell 21 onward to test new configurations.

In [ ]:
# ── Hyperparameters — edit and re-run cells 22 onward ───────────────
# Default = Run 011 (Augmented Baseline) from the CLI experiment.
# Reference: topic-modeling.faculytics/reports/topic_modeling_full_report_runs_001_013.pdf
# Expected: ~19 topics, embedding_coherence ~0.59, diversity ~0.94, outlier ~0.33
params = {
    "min_topic_size":      15,    # HDBSCAN min_cluster_size
    "nr_topics":           20,    # hierarchical merge to 20 (yields 19 after BERTopic accounts for outlier)
    "umap_n_neighbors":    20,    # wider neighborhood preserves local structure at scale
    "umap_n_components":   10,    # higher dims preserve more semantic info for HDBSCAN
    "hdbscan_min_samples": 5,     # outlier sensitivity
    "use_keybert":         True,  # KeyBERTInspired (CLI Run 011 architecture pivot)
}

# ── Reference Run Timeline (CLI Runs 001–013) ─────────────────────
# Mirrors the tuning trajectory documented in the full tuning report.
# Uncomment one block at a time to reproduce a specific run.
# Reported metrics are on the AUGMENTED dataset unless noted.
# `use_keybert=False` reproduces the c-TF-IDF era (Runs 001–010);
# `True` is the KeyBERTInspired pivot locked in at Run 011.

# ── Phase 1 — Baseline & topic-count probing (c-TF-IDF) ─────────────

# Run 001 — Baseline, no preprocessing, default BERTopic params
#   → 78 topics, diversity 0.8013, outliers 0.4239 (no embed_coh metric yet)
#   → too granular; c-TF-IDF dominated by 'maam/sir/teacher' noise
# params = {"min_topic_size": 15, "nr_topics": None, "umap_n_neighbors": 15,
#           "umap_n_components": 5, "hdbscan_min_samples": 5, "use_keybert": False}
#
# Run 002 — Added nr_topics=20 to force hierarchical merge
#   → 38 topics, diversity 0.8447, outliers 0.4290; NPMI went negative
# params = {"min_topic_size": 15, "nr_topics": 20, "umap_n_neighbors": 15,
#           "umap_n_components": 5, "hdbscan_min_samples": 5, "use_keybert": False}
#
# Run 003 — Reduced min_topic_size to 10 (no nr_topics constraint)
#   → 82 topics, diversity 0.8366, outliers 0.3297
#   → too granular; confirmed nr_topics constraint is necessary
# params = {"min_topic_size": 10, "nr_topics": None, "umap_n_neighbors": 15,
#           "umap_n_components": 5, "hdbscan_min_samples": 5, "use_keybert": False}

# ── Phase 2 — UMAP/HDBSCAN grid search (c-TF-IDF) ─────────────────

# Run 004 — Introduced embedding_coherence; first clean 19-topic result
#   → embed_coh 0.4788, diversity 0.8474, 19 topics, outliers 0.3297
#   → composite score formula established
# params = {"min_topic_size": 10, "nr_topics": 20, "umap_n_neighbors": 15,
#           "umap_n_components": 5, "hdbscan_min_samples": 5, "use_keybert": False}
#
# Run 005 — Grid: target 15 topics, larger min_topic_size
#   → embed_coh 0.4937, diversity 0.8071, 14 topics, outliers 0.3472
#   → topic count below interpretable range; diversity dropped
# params = {"min_topic_size": 20, "nr_topics": 15, "umap_n_neighbors": 15,
#           "umap_n_components": 5, "hdbscan_min_samples": 8, "use_keybert": False}
#
# Run 006 — Grid: wider UMAP (n=20, dim=8), auto topic count
#   → embed_coh 0.4942, diversity 0.8481, 52 topics, outliers 0.3062
#   → 52 topics too many for educators; composite score penalized
# params = {"min_topic_size": 25, "nr_topics": None, "umap_n_neighbors": 20,
#           "umap_n_components": 8, "hdbscan_min_samples": 8, "use_keybert": False}
#
# Run 007 — Grid: aggressive reduction to 10 topics
#   → embed_coh 0.4925, diversity 0.7667, 9 topics, outliers 0.3439
#   → below 10-topic minimum; coherence plateau confirmed
# params = {"min_topic_size": 30, "nr_topics": 10, "umap_n_neighbors": 15,
#           "umap_n_components": 5, "hdbscan_min_samples": 10, "use_keybert": False}

# ── Phase 3 — Stop-word treadmill (params held; c-TF-IDF) ───────────
# Wide UMAP (n=20, dim=10) locked in. Only the stop-word list varied.

# Run 008 — Added role stop words; embedding-coherence bug exposed
#   → embed_coh 0.0000 (bug: LaBSE not loaded) — masked all gains since Run 004
# params = {"min_topic_size": 15, "nr_topics": 20, "umap_n_neighbors": 20,
#           "umap_n_components": 10, "hdbscan_min_samples": 5, "use_keybert": False}
#
# Run 009 — Bug fixed; added Cebuano possessives/fillers to stop list
#   → embed_coh 0.4826, diversity 0.9053, 19 topics — first valid post-bugfix reading
# params = {"min_topic_size": 15, "nr_topics": 20, "umap_n_neighbors": 20,
#           "umap_n_components": 10, "hdbscan_min_samples": 5, "use_keybert": False}
#
# Run 010 — Added English generic verbs/fillers
#   → embed_coh 0.4823, diversity 0.9053 — identical to Run 009; plateau confirmed
#   → stop-word expansion exhausted — architecture change required
# params = {"min_topic_size": 15, "nr_topics": 20, "umap_n_neighbors": 20,
#           "umap_n_components": 10, "hdbscan_min_samples": 5, "use_keybert": False}

# ── Phase 4 — Architecture pivot & production (KeyBERTInspired) ────

# Run 011 ★ ARCHITECTURE PIVOT — KeyBERTInspired replaces c-TF-IDF
#   → embed_coh 0.5968 (+23.7% vs Run 010), diversity 0.9474, 19 topics
#   → AUGMENTED DATASET BASELINE LOCKED — current default above
#
# Run 012 — Validated Run 011 params on real UC dataset (unfiltered)
#   → embed_coh 0.6812, diversity 0.9842, 19 topics, outliers 0.4776 (real-raw)
#   → quality degraded: duplicate 'good teaching' clusters, '10/10' rating noise
# Apply by setting DATASET="real" in cell 4; params unchanged from Run 011.
#
# Run 013 ★ PRODUCTION BASELINE — sentiment-gated filter
#   → embed_coh 0.6495, diversity 0.9789, 19 topics, outliers 0.5202 (real-filtered)
#   → positive <10 words excluded (35.4% of real data) — production locked
# Apply by setting DATASET="real_filtered" in cell 4; params unchanged from Run 011.
params

In [ ]:
# ── Tier 1 toggle ──────────────────────────────────────────────────
# When TIER1_GUIDED=True the build path uses build_bertopic_guided()
# (seed-aspect topics + multi-rep + soft assignment).
# When False, behaves exactly like the existing notebook (Run 011/013).
TIER1_GUIDED = True

# Audit seed coverage on the cleaned corpus *before* fitting.
# An aspect with <50 hits in your data is too narrow — surface, then decide
# whether to expand the seed list or drop the aspect.
if TIER1_GUIDED:
    coverage = lib.audit_aspect_coverage(texts)
    cov_df = (
        pd.DataFrame(
            [(k, v, v / len(texts) * 100) for k, v in coverage.items()],
            columns=["aspect", "doc_count", "pct"],
        )
        .sort_values("doc_count", ascending=False)
        .reset_index(drop=True)
    )
    print("aspect coverage on this corpus:")
    print(cov_df.to_string(index=False))

    weak = [a for a, c in coverage.items() if c < 50]
    if weak:
        print(f"\n⚠ aspects with <50 hits — review seed lists: {weak}")


In [ ]:
if TIER1_GUIDED:
    topic_model = lib.build_bertopic_guided(params, embedding_model=embed_model)
else:
    topic_model = lib.build_bertopic(params, embedding_model=embed_model)

# Re-seed immediately before fit (matches src/topic_model.py:147)
np.random.seed(SEED)
topics, probs = topic_model.fit_transform(texts, embeddings=embeddings)
topics = list(topics)


In [ ]:
n_topics = len(set(topics)) - (1 if -1 in topics else 0)
n_outliers = sum(1 for t in topics if t == -1)
print(f"topics:    {n_topics}")
print(f"outliers:  {n_outliers:,} ({n_outliers / len(topics):.1%})")
print()
topic_model.get_topic_info().head(15)

## 6. Topic Visualizations

### Intertopic distance map

Each bubble is one topic, sized by document count. Topics close together are semantically similar; far apart = distinct themes. The 2D layout is a UMAP projection of topic c-TF-IDF representations.

**Watch for:** overlapping or stacked bubbles → redundant topics that could be merged. Well-spaced bubbles → good topic separation.

In [ ]:
# Intertopic distance map (interactive plotly)
fig_topics = topic_model.visualize_topics()
fig_topics.write_html(str(FIG_DIR / "intertopic_distance.html"))
fig_topics.show()

### Top keywords per topic

Top 8 keywords for the 12 largest topics, ordered by KeyBERTInspired's relevance score (cosine similarity to the cluster centroid in LaBSE space).

**Watch for:** keywords within a single topic that don't seem related (low intra-topic coherence), or many topics sharing the same keywords (low diversity). A good topic reads like a small theme — e.g., *"grading, fair, criteria, transparent, system"*.

In [ ]:
fig_bar = topic_model.visualize_barchart(top_n_topics=12, n_words=8)
fig_bar.write_html(str(FIG_DIR / "barchart.html"))
fig_bar.show()

### Topic hierarchy

A dendrogram showing how topics would merge if you forced fewer clusters. Read it left-to-right: topics joining at a low distance are highly similar; long horizontal branches before joining = distinct themes.

**Watch for:** pairs of topics joining very close to the leaves — those are essentially the same topic and could be merged by lowering `nr_topics`.

In [ ]:
fig_hier = topic_model.visualize_hierarchy()
fig_hier.write_html(str(FIG_DIR / "hierarchy.html"))
fig_hier.show()

### Document map

Same 2D projection as the embedding scatter, but every dot is now colored by its assigned topic. Hover over points to read the original feedback.

**Watch for:** clean separated color regions = good clustering. A large grey cloud = high outlier rate (HDBSCAN's `-1`); ~33% on the augmented dataset is expected. Color regions that interleave heavily = topics aren't well separated in embedding space.

In [ ]:
# Reuse the cell-19 2D projection so we don't recompute UMAP
fig_docs = topic_model.visualize_documents(
    texts,
    reduced_embeddings=reduced_2d,
    embeddings=embeddings,
)
fig_docs.write_html(str(FIG_DIR / "document_map.html"))
fig_docs.show()

### Topic-similarity heatmap

Pairwise cosine similarity between topic c-TF-IDF vectors. Diagonal is always 1.0 (a topic compared to itself). Off-diagonal cells show how similar two different topics are.

**Watch for:** bright off-diagonal cells = topic redundancy (good candidates for merging). A heatmap that's mostly dark away from the diagonal = well-differentiated topics.

In [ ]:
fig_heat = topic_model.visualize_heatmap()
fig_heat.write_html(str(FIG_DIR / "heatmap.html"))
fig_heat.show()

## 6.5 Multi-topic & aspect labels (Tier 1)

Two outputs only available when `TIER1_GUIDED=True`:

- **Multi-topic assignment** — `approximate_distribution()` gives every
  document a probability over topics. Above `primary_threshold` becomes
  the primary topic; if a second topic clears `secondary_threshold`
  *and* the primary–secondary gap is small, it's a multi-topic doc.

- **Aspect mapping** — each discovered topic is matched to the closest
  academic aspect by comparing its keyword centroid to each aspect's
  seed-word centroid in LaBSE space. Topics whose closest aspect is
  below `match_threshold` are labeled `emergent`.

Tune the thresholds on the printed distribution stats, then commit.


In [ ]:
if TIER1_GUIDED:
    # 1) Aspect mapping — per topic
    aspect_mapping = lib.map_topics_to_aspects(
        topic_model,
        embed_model,
        match_threshold=0.65, #was 0.50
        representation="Main",
    )

    # 2) Multi-topic assignment — per document
    multi = lib.assign_multi_topic(
        topic_model,
        texts,
        primary_threshold=0.20,
        secondary_threshold=0.15,
        secondary_gap_max=0.20, # was 0.30
    )
    multi_df = pd.DataFrame(multi)
    multi_df["text"] = [t[:120] + ("..." if len(t) > 120 else "") for t in texts]

    # Sanity: how often did we actually produce a secondary topic?
    n_multi   = int(multi_df["is_multi_topic"].sum())
    n_outlier = int((multi_df["primary_topic"] == -1).sum())
    print(f"multi-topic rate: {n_multi:,}/{len(multi_df):,} ({n_multi/len(multi_df):.1%})")
    print(f"outlier rate (soft): {n_outlier:,}/{len(multi_df):,} ({n_outlier/len(multi_df):.1%})")
    print(f"hard outlier rate (HDBSCAN -1): {sum(1 for t in topics if t == -1)/len(topics):.1%}")

    # 3) Per-topic table with aspect labels
    topic_info_multi = lib.extract_topic_info_multi(topic_model, aspect_mapping)

    asp_df = pd.DataFrame([
        {
            "topic_id":          t["topic_id"],
            "doc_count":         t["doc_count"],
            "aspect":            t.get("aspect_label", "n/a"),
            "aspect_similarity": t.get("aspect_similarity", 0.0),
            "is_emergent":       t.get("is_emergent", False),
            "keywords_main":     ", ".join(t["keywords"][:5]),
            "keywords_mmr":      ", ".join(t.get("keywords_mmr", [])[:5]),
        }
        for t in topic_info_multi
    ]).sort_values(["aspect", "doc_count"], ascending=[True, False]).reset_index(drop=True)

    from IPython.display import display
    display(
        asp_df.style.background_gradient(subset=["aspect_similarity"], cmap="RdYlGn",
                                         vmin=0.4, vmax=0.85)
                    .background_gradient(subset=["doc_count"], cmap="Blues")
    )


## 7. Evaluate

**Embedding coherence** (primary, target > 0.5) — average pairwise cosine similarity between top-10 keyword embeddings per topic. Language-agnostic; works on multilingual data.

**Topic diversity** (target > 0.7) — fraction of unique words across all topic top-10s.

**Outlier ratio** (target < 20%) — fraction of documents in topic `-1`. Often 33% on augmented and 47–52% on real data — accepted limitation per CLI Run 013 (HDBSCAN inherently rejects unclusterable short texts).

**Silhouette** — cluster separation in embedding space (higher better, computed with `metric=cosine` excluding outliers).

**NPMI** (reference only, target > 0.1) — co-occurrence-based; systematically fails on multilingual corpora because semantically equivalent words across languages don't co-occur (Hoyle et al. 2021).

In [ ]:
metrics = lib.compute_metrics(
    topic_model, topics, texts, embeddings, embed_model=embed_model,
)
metrics

### Metrics vs targets

Each bar is one metric; the dashed black line marks its target. Green = passing, red = failing. `outlier_ratio` is the only metric where lower is better — it passes when below `0.2`.

**Read it as:** if all bars are green except `outlier_ratio` and `npmi_coherence`, the run is healthy. NPMI failing is expected on multilingual data (kept here for reference); high outlier rate is data-inherent for HDBSCAN on short Filipino-English feedback.

In [ ]:
# Metrics-vs-targets bar chart
metric_targets = {
    "embedding_coherence": 0.5,
    "topic_diversity":     0.7,
    "outlier_ratio":       0.2,   # lower is better — special-cased below
    "npmi_coherence":      0.1,
    "silhouette_score":    0.0,
}
names  = list(metric_targets.keys())
values = [metrics[n] for n in names]
targets = [metric_targets[n] for n in names]

def _passes(name, value, target):
    return (value <= target) if name == "outlier_ratio" else (value >= target)

colors = ["#55A868" if _passes(n, v, t) else "#C44E52" for n, v, t in zip(names, values, targets)]

fig, ax = plt.subplots(figsize=(10, 5))
y_pos = np.arange(len(names))
ax.barh(y_pos, values, color=colors, edgecolor="white")
for y, t in zip(y_pos, targets):
    ax.plot([t, t], [y - 0.4, y + 0.4], color="black", linestyle="--", linewidth=1.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(names)
ax.set_xlabel("value")
ax.set_title(f"metrics vs targets — {RUN_ID} (dashed = target; outlier_ratio: lower is better)")
for y, v in zip(y_pos, values):
    ax.text(v, y, f"  {v:.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

### Per-topic quality table

Same embedding-coherence calculation as the headline metric, but computed per topic instead of averaged. Sorted by document count (largest topics on top). Background gradient highlights weak topics: red `coherence` cells indicate topics whose top keywords don't sit close together in LaBSE space — review their keywords to decide whether to merge or drop.

**Rule of thumb:** topics with `coherence < 0.4` are usually noise clusters. Topics above `0.6` are typically clean themes.

In [ ]:
# Per-topic quality table — flag weak topics by intra-topic embedding coherence
per_topic_coh = lib.compute_per_topic_coherence(topic_model, embed_model)
topic_info = lib.extract_topic_info(topic_model)

rows = []
for t in topic_info:
    rows.append({
        "topic_id":    t["topic_id"],
        "doc_count":   t["doc_count"],
        "top_5_keywords": ", ".join(t["keywords"][:5]),
        "coherence":   per_topic_coh.get(t["topic_id"], 0.0),
    })
topic_df = pd.DataFrame(rows).sort_values("doc_count", ascending=False).reset_index(drop=True)

topic_df.style.background_gradient(subset=["coherence"], cmap="RdYlGn", vmin=0.3, vmax=0.8) \
             .background_gradient(subset=["doc_count"], cmap="Blues")

## 8. Save Run Artifacts

In [ ]:
lib.save_run_artifacts(
    RUN_DIR,
    params=params,
    metrics=metrics,
    topic_info=topic_info,
    model=topic_model,
    n_texts=len(texts),
)
print(f"\nartifacts in {RUN_DIR}:")
for p in sorted(RUN_DIR.iterdir()):
    print(f"  {p.name}")

In [ ]:
# ── Tier 1 sidecar artifacts (only when TIER1_GUIDED) ────────────────
if TIER1_GUIDED:
    # multi_topic_assignments.csv — one row per doc
    multi_path = RUN_DIR / "multi_topic_assignments.csv"
    multi_df.to_csv(multi_path, index=False)

    # aspect_mapping.json — topic_id → (aspect, similarity)
    asp_map_path = RUN_DIR / "aspect_mapping.json"
    with open(asp_map_path, "w") as f:
        json.dump(
            {str(k): {"aspect": v[0], "similarity": v[1]}
             for k, v in aspect_mapping.items()},
            f, indent=2,
        )

    # topic_info_multi.json — per-topic with aspect + keywords_main + keywords_mmr
    topic_info_path = RUN_DIR / "topic_info_multi.json"
    with open(topic_info_path, "w") as f:
        json.dump(topic_info_multi, f, indent=2, default=str)

    print("\nTier 1 sidecar artifacts:")
    print(f"  {multi_path.name}")
    print(f"  {asp_map_path.name}")
    print(f"  {topic_info_path.name}")


In [ ]:
# Native-speaker spot-check (Tier 1 gate #2)
if TIER1_GUIDED:
    SAMPLES_PER_ASPECT = 30

    doc_view = multi_df.copy()
    doc_view["aspect"] = doc_view["primary_topic"].map(
        lambda t: aspect_mapping.get(t, ("unknown", 0.0))[0]
    )
    # NEW: human-readable secondary aspect
    doc_view["secondary_aspect"] = doc_view["secondary_topic"].map(
        lambda t: aspect_mapping.get(int(t), ("unknown", 0.0))[0]
                  if pd.notna(t) else ""
    )

    samples = []
    for aspect_name, group in doc_view.groupby("aspect"):
        n = min(SAMPLES_PER_ASPECT, len(group))
        samples.append(group.sample(n=n, random_state=42))
    sample_df = pd.concat(samples).sort_values(
        ["aspect", "primary_confidence"], ascending=[True, False]
    )

    review_df = sample_df[[
        "aspect", "secondary_aspect", "is_multi_topic",
        "primary_topic", "primary_confidence", "secondary_topic",
        "text"
    ]].reset_index(drop=True)
    review_df["verdict"] = ""
    review_df["note"] = ""
    review_path = RUN_DIR / "spot_check_samples.csv"
    review_df.to_csv(review_path, index=False)
    print(f"saved {len(review_df)} samples across {review_df['aspect'].nunique()} aspects to {review_path.name}")
    review_df.head(20)

In [ ]:
# Confirm figures were saved
print(f"figures in {FIG_DIR}:")
for p in sorted(FIG_DIR.iterdir()):
    print(f"  {p.name} ({p.stat().st_size / 1e3:.1f} KB)")

## End

To explore alternative hyperparameters in a sweep, see [`grid_search.ipynb`](./grid_search.ipynb). It runs the same 4 configurations as the CLI's `auto_tune.py` and ranks them by composite score.

To re-run with different params: change cell 21 (`params` dict), then re-run from cell 22 onward (embeddings stay cached).

To re-run on a different dataset: change `DATASET` in cell 4, then run all cells (a new embeddings cache is created automatically).